
# Model Training — Insurance Premium Prediction

**Objective**: Train and compare baseline (Ridge Regression) and advanced (XGBoost) models 
to predict insurance charges, tracking all experiments with MLflow.

## 1. Setup & Imports

Loading libraries for modeling (`scikit-learn`, `xgboost`), experiment tracking (`mlflow`), 
and evaluation metrics. Also importing our custom feature engineering functions from `src/`.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import sys

sys.path.append('..')
from src.features.build_features import build_features, split_data

# Set MLflow tracking using SQLite (file-based tracking is deprecated in newer MLflow)
project_root = os.path.abspath('..')
db_path = os.path.join(project_root, 'mlflow.db')

mlflow.set_tracking_uri(f"sqlite:///{db_path.replace(os.sep, '/')}")
mlflow.set_experiment("insurance-charges-prediction")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

MLflow tracking URI: sqlite:///c:/Users/Ali Raza/Desktop/inusrance-premium-mlops/mlflow.db


## 2. Prepare Data for Training

Loading the dataset, applying feature engineering, and splitting into train/test sets.

In [2]:
# Load raw data
df = pd.read_csv('../data/raw/insurance.csv')
df = df.drop_duplicates()

#Apply feature engineering kjk

df_ready = build_features(df)

#Split into train/test
X_train, X_test, y_train,y_test = split_data(df_ready)
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

Training set: (1069, 9)
Testing set: (268, 9)


## 3. Baseline Model — Ridge Regression

Training a simple Ridge Regression as our baseline. This gives us a reference point 
to compare more complex models against. All parameters and metrics are logged to MLflow.

In [3]:
with mlflow.start_run(run_name="ridge_baseline"):
    # Define model
    alpha = 1.0
    model = Ridge(alpha=alpha)

    # Train
    model.fit(X_train,y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Log to ML flow 
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", alpha)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, "model")

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")
    
  

2026/07/28 11:07:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MAE: 2817.98
RMSE: 4555.34
R2 Score: 0.8871


In [4]:
import mlflow.xgboost

## 4. Advanced Model — XGBoost

Training an XGBoost model to compare against the Ridge baseline. XGBoost typically 
handles non-linear relationships and feature interactions better than linear models.

In [5]:
with mlflow.start_run(run_name="xgboost_v1"):
    
    # Define model
    n_estimators = 100
    max_depth = 4
    learning_rate = 0.1
    
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42
    )
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Log to MLflow
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")

2026/07/28 11:11:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MAE: 2519.76
RMSE: 4327.10
R2 Score: 0.8981


In [6]:
from sklearn.linear_model import LinearRegression,Lasso

## 5. Linear Regression (No Regularization)

Training a plain Linear Regression to compare against Ridge — this tests whether 
regularization is actually helping, or if the simplest possible model performs just as well.

In [7]:
with mlflow.start_run(run_name="linear_regression"):
    
    model = LinearRegression()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, "model")
    
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")

2026/07/28 11:18:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MAE: 2828.97
RMSE: 4572.81
R2 Score: 0.8862


## 6. Lasso Regression

Training Lasso, which can shrink weak feature coefficients to exactly zero — 
useful for confirming which features (like region) may be unnecessary.

In [9]:
with mlflow.start_run(run_name="lasso"):
    
    alpha = 1.0
    model = Lasso(alpha=alpha)
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model_type", "Lasso")
    mlflow.log_param("alpha", alpha)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(model, "model")
    
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.4f}")
    
    # Bonus: check which features Lasso zeroed out
    print("\nFeature Coefficients:")
    for feature, coef in zip(X_train.columns, model.coef_):
        print(f"  {feature}: {coef:.2f}")

2026/07/28 11:19:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MAE: 2828.51
RMSE: 4572.06
R2 Score: 0.8862

Feature Coefficients:
  age: 259.03
  sex: 641.50
  bmi: 13.41
  children: 569.17
  smoker: -21514.66
  region_northwest: -420.85
  region_southeast: -878.35
  region_southwest: -866.37
  smoker_bmi_interaction: 1471.10


In [10]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/xgboost_model.pkl')
print("Model saved locally at models/xgboost_model.pkl")

Model saved locally at models/xgboost_model.pkl
